In [1]:
print('Hello World')

Hello World


Create SparkSession

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MovieLens").getOrCreate()
print("Spark version:", spark.version)


Spark version: 4.1.1


In [22]:
from pyspark.sql.functions import col, count, sum as spark_sum, lit, log1p, explode, mean
from pathlib import Path

In [23]:

bronze_db = Path("../database/bronze")
silver_db = Path("../database/silver")
gold_db   = Path("../database/gold")

In [3]:
movies  = spark.read.parquet(str(silver_db/"movies"))

print(movies.show(5))

+-------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|movieId|               title|              genres|      load_timestamp|          movieTitle|year|         genres_list|
+-------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|2026-05-20 00:14:...|           Toy Story|1995|[Adventure, Anima...|
|      2|      Jumanji (1995)|Adventure|Childre...|2026-05-20 00:14:...|             Jumanji|1995|[Adventure, Child...|
|      3|Grumpier Old Men ...|      Comedy|Romance|2026-05-20 00:14:...|    Grumpier Old Men|1995|   [Comedy, Romance]|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|2026-05-20 00:14:...|   Waiting to Exhale|1995|[Comedy, Drama, R...|
|      5|Father of the Bri...|              Comedy|2026-05-20 00:14:...|Father of the Bri...|1995|            [Comedy]|
+-------+--------------------+----------

In [9]:
ratings  = spark.read.parquet(str(silver_db/"ratings"))

ratings.select("rating_year").distinct().sort(["rating_year"]).show(100)

+-----------+
|rating_year|
+-----------+
|       1996|
|       1997|
|       1998|
|       1999|
|       2000|
|       2001|
|       2002|
|       2003|
|       2004|
|       2005|
|       2006|
|       2007|
|       2008|
|       2009|
|       2010|
|       2011|
|       2012|
|       2013|
|       2014|
|       2015|
|       2016|
|       2017|
|       2018|
+-----------+



In [20]:
movie_gold = spark.read.parquet(str(gold_db / "movies_gold"))
movie_genre_gold = spark.read.parquet(str(gold_db / "movie_genre_gold"))

In [25]:
candidates = movie_gold.filter(col("rating_count") >= 10)
print(candidates.sort(col("rating_count").desc()).show())

+-------+--------------------+------------------+------------+---------+------------------+
|movieId|          movieTitle|        avg_rating|rating_count|tag_count|        rating_std|
+-------+--------------------+------------------+------------+---------+------------------+
|    356|        Forrest Gump| 4.164133738601782|         329|        9|0.8312435184333592|
|    318|Shawshank Redempt...| 4.429022082018944|         317|        4|0.7130186250284273|
|    296|        Pulp Fiction| 4.197068403908814|         307|      181|0.9519971466349181|
|    593|Silence of the La...| 4.161290322580605|         279|        6|0.8539833357342462|
|   2571|         Matrix, The| 4.192446043165519|         278|        5|0.9752425167066221|
|    260|Star Wars: Episod...|  4.23107569721109|         251|       26|0.8720036916475897|
|    480|       Jurassic Park|              3.75|         238|        1|0.8553011658990147|
|    110|          Braveheart| 4.031645569620227|         237|       10|0.967726

In [ ]:
print(movie_gold.show())

In [ ]:
print(movie_genre_gold.show())



+-------+--------------------+---------+------+-----------+------------+
|movieId|          movieTitle|    genre|rating|rating_year|rating_month|
+-------+--------------------+---------+------+-----------+------------+
|   1009|Escape to Witch M...|Adventure|   3.0|       2000|           7|
|   1009|Escape to Witch M...| Children|   3.0|       2000|           7|
|   1009|Escape to Witch M...|  Fantasy|   3.0|       2000|           7|
|    101|       Bottle Rocket|Adventure|   5.0|       2000|           7|
|    101|       Bottle Rocket|   Comedy|   5.0|       2000|           7|
|    101|       Bottle Rocket|    Crime|   5.0|       2000|           7|
|    101|       Bottle Rocket|  Romance|   5.0|       2000|           7|
|   1023|Winnie the Pooh a...|Animation|   5.0|       2000|           7|
|   1023|Winnie the Pooh a...| Children|   5.0|       2000|           7|
|   1023|Winnie the Pooh a...|  Musical|   5.0|       2000|           7|
|   1024|Three Caballeros,...|Animation|   5.0|    

In [ ]:
df.printSchema()


root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [10]:
df.groupBy("movieId").count().show(5)

+-------+-----+
|movieId|count|
+-------+-----+
|   1580|  165|
|   2366|   25|
|   3175|   75|
|   1088|   42|
|  32460|    4|
+-------+-----+
only showing top 5 rows
